In [25]:
import pandas as pd
import numpy as np

In [26]:
df= pd.read_csv("nifty50_top20_1d.csv")
df.head()

,Stock_Name,Datetime,Open,High,Low,Close,Volume
0,SBIN,2025-05-29 00:00:00,799.000000,800.400024,791.349976,797.349976,14034127
1,SBIN,2025-05-30 00:00:00,798.000000,814.500000,792.599976,812.299988,22037314
2,SBIN,2025-06-02 00:00:00,812.849976,822.549988,809.200012,813.650024,26891575
3,SBIN,2025-06-03 00:00:00,816.200012,818.400024,805.150024,809.799988,14247597
4,SBIN,2025-06-04 00:00:00,817.000000,817.000000,805.250000,806.500000,9607927


In [40]:
import pandas as pd
import numpy as np

# Datetime
datetime_col = "DateTime" if "DateTime" in df.columns else "Datetime"
df[datetime_col] = pd.to_datetime(df[datetime_col])

# Returns
df["Return_1d"] = df["Close"].pct_change(1)
df["Return_5d"] = df["Close"].pct_change(5)
df["Return_20d"] = df["Close"].pct_change(20)

# Volatility
df["Volatility_20d"] = df["Return_1d"].rolling(20).std()

# Volume Features
df["Volume_Change"] = df["Volume"].pct_change()
df["Volume_SMA_20"] = df["Volume"].rolling(20).mean()

# EMA Features
df["EMA20"] = df["Close"].ewm(span=20, adjust=False).mean()
df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()
df["EMA_Spread"] = df["EMA20"] - df["EMA50"]

# Price Above EMA20
df["Price_Above_Below_EMA20"] = (
    df["Close"] > df["EMA20"]
).astype(int)

# RSI
delta = df["Close"].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss
df["RSI_14"] = 100 - (100 / (1 + rs))

# MACD
ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()

df["MACD"] = ema12 - ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
df["MACD_Histogram"] = df["MACD"] - df["MACD_Signal"]

# Bollinger Bands
sma20 = df["Close"].rolling(20).mean()
std20 = df["Close"].rolling(20).std()

df["Bollinger_Upper"] = sma20 + (2 * std20)
df["Bollinger_Lower"] = sma20 - (2 * std20)
df["Bollinger_Band_Width"] = (
    df["Bollinger_Upper"] - df["Bollinger_Lower"]
)

# ATR 14
high_low = df["High"] - df["Low"]
high_close = (df["High"] - df["Close"].shift()).abs()
low_close = (df["Low"] - df["Close"].shift()).abs()

tr = pd.concat(
    [high_low, high_close, low_close],
    axis=1
).max(axis=1)

df["ATR_14"] = tr.rolling(14).mean()

# Time Features
df["Day_of_Week"] = df[datetime_col].dt.dayofweek
df["Month"] = df[datetime_col].dt.month

# Lag Features
df["Lagged_Close_1"] = df["Close"].shift(1)
df["Lagged_Close_5"] = df["Close"].shift(5)

# Target (Up/Down Tomorrow)
df["Target"] = (
    df["Close"].shift(-1) > df["Close"]
).astype(int)

# Returns
df["Return_1d"] = df["Close"].pct_change(1)
df["Return_5d"] = df["Close"].pct_change(5)
df["Return_20d"] = df["Close"].pct_change(20)

# Volatility
df["Volatility_20d"] = df["Close"].pct_change().rolling(20).std()

# Volume Features
df["Volume_Change"] = df["Volume"].pct_change()
df["Volume_SMA_20"] = df["Volume"].rolling(20).mean()

# EMA Spread
df["EMA_Spread"] = df["EMA20"] - df["EMA50"]

# Price Above EMA20
df["Price_Above_Below_EMA20"] = (
    df["Close"] > df["EMA20"]
).astype(int)

# ATR 14
high_low = df["High"] - df["Low"]
high_close = (df["High"] - df["Close"].shift()).abs()
low_close = (df["Low"] - df["Close"].shift()).abs()

tr = pd.concat(
    [high_low, high_close, low_close],
    axis=1
).max(axis=1)

df["ATR_14"] = tr.rolling(14).mean()

# Lag Features
df["Lagged_Close_1"] = df["Close"].shift(1)
df["Lagged_Close_5"] = df["Close"].shift(5)


# Optional Target (Tomorrow Up/Down)
df["Target"] = (
    df["Close"].shift(-1) > df["Close"]
).astype(int)

# Remove NaN
df.dropna(inplace=True)

# Save Final Dataset
df.to_csv("stock_features.csv", index=False)

print("Shape:", df.shape)
print("Columns:", len(df.columns))
print(df.columns.tolist())

Shape: (4442, 31)
Columns: 31
['Stock_Name', 'Datetime', 'Open', 'High', 'Low', 'Close', 'Volume', 'Return_1d', 'Return_5d', 'Return_20d', 'Volatility_20d', 'Volume_Change', 'Volume_SMA_20', 'EMA20', 'EMA50', 'EMA_Spread', 'Price_Above_Below_EMA20', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'Bollinger_Upper', 'Bollinger_Lower', 'Bollinger_Band_Width', 'ATR_14', 'Day_of_Week', 'Month', 'Lagged_Close_1', 'Lagged_Close_5', 'Target', 'Stock_TE']


In [43]:
df["Year"] = df["Datetime"].dt.year
df["Month"] = df["Datetime"].dt.month
df["Day"] = df["Datetime"].dt.day
df["Day_of_Week"] = df["Datetime"].dt.dayofweek
df["Quarter"] = df["Datetime"].dt.quarter

In [44]:
# Target Encoding

stock_target_mean = (
    df.groupby("Stock_Name")["Target"]
      .mean()
)

df["Stock_TE"] = (
    df["Stock_Name"]
      .map(stock_target_mean)
)

print(df[['Stock_Name', 'Datetime', 'Open', 'High', 'Low', 'Close', 'Volume', 'Return_1d', 'Return_5d', 'Return_20d', 'Volatility_20d', 'Volume_Change', 'Volume_SMA_20', 'EMA20', 'EMA50', 'EMA_Spread', 'Price_Above_Below_EMA20', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'Bollinger_Upper', 'Bollinger_Lower', 'Bollinger_Band_Width', 'ATR_14', 'Day_of_Week', 'Month', 'Lagged_Close_1', 'Lagged_Close_5', 'Target', 'Stock_TE']].head())

   Stock_Name   Datetime        Open        High         Low       Close  \
40       SBIN 2025-07-24  820.000000  821.950012  810.650024  815.700012   
41       SBIN 2025-07-25  815.000000  819.200012  805.500000  806.549988   
42       SBIN 2025-07-28  806.549988  808.950012  796.000000  797.150024   
43       SBIN 2025-07-29  796.200012  800.000000  793.099976  799.200012   
44       SBIN 2025-07-30  798.950012  803.400024  796.200012  801.650024   

      Volume  Return_1d  Return_5d  Return_20d  ...  Bollinger_Upper  \
40  10118309  -0.006032  -0.016043    0.023399  ...       830.712545   
41   6734439  -0.011217  -0.020404    0.001428  ...       830.617365   
42   7734158  -0.011655  -0.032820   -0.028281  ...       831.355227   
43   7833759   0.002572  -0.019386   -0.025722  ...       831.345628   
44   8743723   0.003066  -0.023152   -0.013839  ...       831.528065   

    Bollinger_Lower  Bollinger_Band_Width     ATR_14  Day_of_Week  Month  \
40       800.277457             30

In [45]:
df

,Stock_Name,Datetime,Open,High,Low,Close,Volume,Return_1d,Return_5d,Return_20d,...,ATR_14,Day_of_Week,Month,Lagged_Close_1,Lagged_Close_5,Target,Stock_TE,Year,Day,Quarter
40,SBIN,2025-07-24,820.000000,821.950012,810.650024,815.700012,10118309,-0.006032,-0.016043,0.023399,...,10.821433,3,7,820.650024,829.000000,0,0.53110,2025,24,3
41,SBIN,2025-07-25,815.000000,819.200012,805.500000,806.549988,6734439,-0.011217,-0.020404,0.001428,...,11.264291,4,7,815.700012,823.349976,0,0.53110,2025,25,3
42,SBIN,2025-07-28,806.549988,808.950012,796.000000,797.150024,7734158,-0.011655,-0.032820,-0.028281,...,11.603577,0,7,806.549988,824.200012,1,0.53110,2025,28,3
43,SBIN,2025-07-29,796.200012,800.000000,793.099976,799.200012,7833759,0.002572,-0.019386,-0.025722,...,11.835720,1,7,797.150024,815.000000,1,0.53110,2025,29,3
44,SBIN,2025-07-30,798.950012,803.400024,796.200012,801.650024,8743723,0.003066,-0.023152,-0.013839,...,11.735722,2,7,799.200012,820.650024,0,0.53110,2025,30,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4477,TITAN,2026-05-22,4057.199951,4113.799805,4054.100098,4079.800049,768112,-0.000808,-0.021420,-0.074875,...,134.257150,4,5,4083.100098,4169.100098,1,0.48996,2026,22,2
4478,TITAN,2026-05-25,4120.100098,4170.000000,4110.399902,4159.200195,606086,0.019462,-0.002518,-0.063623,...,135.935704,0,5,4079.800049,4169.700195,0,0.48996,2026,25,2
4479,TITAN,2026-05-26,4151.899902,4160.000000,4091.000000,4105.899902,766717,-0.012815,0.000951,-0.070432,...,130.578561,1,5,4159.200195,4102.000000,1,0.48996,2026,26,2
4480,TITAN,2026-05-27,4098.299805,4165.100098,4091.100098,4137.899902,474044,0.007794,0.007671,-0.067999,...,128.821411,2,5,4105.899902,4106.399902,1,0.48996,2026,27,2


In [35]:

from sklearn.model_selection import TimeSeriesSplit
X = df.drop(columns=["Target"])
y = df["Target"]
tscv = TimeSeriesSplit(n_splits=5)

for train_index, test_index in tscv.split(X):

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    print(
        "Train:", X_train.shape,
        "Test:", X_test.shape
    )

Train: (747, 30) Test: (743, 30)
Train: (1490, 30) Test: (743, 30)
Train: (2233, 30) Test: (743, 30)
Train: (2976, 30) Test: (743, 30)
Train: (3719, 30) Test: (743, 30)


In [52]:
df["Volume_Change"] = (
    df["Volume"]
      .replace(0, np.nan)
      .pct_change()
      .fillna(0)
)
import numpy as np

print(np.isinf(df["Volume_Change"]).sum())

0


In [61]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
scores = cross_val_score(
    model,
    X_scaled,
    y,
    cv=tscv,
    scoring="accuracy"
)


In [62]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

tscv = TimeSeriesSplit(n_splits=5)

scores = cross_val_score(
    model,
    X,
    y,
    cv=tscv,
    scoring="accuracy"
)

print("Fold Scores:", scores)
print("Mean Accuracy:", scores.mean())

Fold Scores: [0.56621622 0.52432432 0.51756757 0.54324324 0.57297297]
Mean Accuracy: 0.5448648648648649


In [63]:
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="accuracy"
)

print("Fold Scores:", scores)
print("Mean Accuracy:", scores.mean())

Fold Scores: [0.55945946 0.54324324 0.55675676 0.58378378 0.60135135]
Mean Accuracy: 0.5689189189189189


In [64]:
import joblib

joblib.dump(model, "stock_model.pkl")

['stock_model.pkl']

In [66]:
from fastapi import FastAPI
import joblib
import pandas as pd

app = FastAPI()

model = joblib.load("stock_model.pkl")

@app.post("/predict")
def predict(data: dict):

    df = pd.DataFrame([data])

    prediction = model.predict(df)[0]

    probability = model.predict_proba(df)[0][1]

    return {
        "prediction": int(prediction),
        "confidence": float(probability)
    }

SyntaxError: invalid syntax (1903356396.py, line 1)